# Cardiac Drift Project — Cheatsheet Weeks 1–3

Reference for the most important Python, pandas, and numpy concepts used across the first three weeks. Each section shows the function or concept, explains what it does and why, and gives a minimal working example.

---

## 1. Reading a .fit file with fitparse

`.fit` files are binary — you cannot open them as text. The `fitparse` library decodes them into messages. Each message has a type (like `record`, `event`, `hrv`) and contains fields (like `heart_rate`, `power`, `timestamp`).

In [ ]:
from fitparse import FitFile

fit_file = FitFile('your_activity.fit')

# List all message types (excluding unknowns)
msg_types = set(m.name for m in fit_file.messages if m.name and 'unknown' not in m.name)
print(msg_types)
# Example output: {'record', 'event', 'hrv', 'session', 'lap', ...}

## 2. Extracting messages into a list of dicts

Each message contains fields. We extract them into a list of dicts so pandas can convert them into a DataFrame.

The dict comprehension `{field.name: field.value for field in message}` maps each field name to its value for one message.

In [ ]:
def get_records(fit_file, record_type, amount=None):
    all_records = []
    for count, message in enumerate(fit_file.get_messages(record_type)):
        if amount is not None and count >= amount:
            break
        field_content = {
            field.name: field.value
            for field in message if field.name and 'unknown' not in field.name
        }
        all_records.append(field_content)
    return all_records

# Returns a list like:
# [{'heart_rate': 108, 'power': 120, 'timestamp': datetime(...)}, ...]

## 3. Creating a DataFrame and setting the timestamp as index

A DataFrame is a table. By default its index is numeric (0, 1, 2...). Setting the timestamp as the index lets you slice by time directly.

**Important:** `set_index()` does not modify the original DataFrame — it returns a new one. You must reassign.

In [ ]:
import pandas as pd

df = pd.DataFrame(get_records(fit_file, 'record'))
df = df.set_index('timestamp')  # reassign — set_index returns a new DataFrame

# Verify the index is in chronological order
print(df.index.is_monotonic_increasing)  # True

# Slice by time range using .loc
# If a timestamp doesn't exist exactly, pandas clamps to the nearest valid one — no error
subset = df.loc['2026-03-08 13:30:00':'2026-03-08 13:45:00']

## 4. Inspecting a DataFrame

In [ ]:
df.info()      # column names, types, non-null counts — shows where NaNs are
df.describe()  # min, max, mean, std, quartiles for every numeric column
df.head(5)     # first 5 rows
df.shape       # (rows, columns) as a tuple

## 5. Finding missing values (NaN)

`isna()` returns True where a value is missing. Combining with boolean indexing extracts just the affected rows.

In [ ]:
# Timestamps where heart_rate is missing
missing_hr = df[df['heart_rate'].isna()].index

# Rows where ALL columns in a set are missing
cols = ['heart_rate', 'power']
missing_both = df[df[cols].isna().all(axis=1)].index

# Check if two sets of timestamps overlap
matches = missing_hr.intersection(some_other_index)

## 6. Forward-fill — handling sensor dropouts

`ffill()` fills a NaN by carrying the last known value forward. The `limit` parameter caps how many consecutive NaNs get filled.

**Why limit matters:** filling 3 seconds of dropped HR is safe. Filling 40 seconds across a pause would fabricate data.

In [ ]:
df['heart_rate'] = df['heart_rate'].ffill(limit=3)
df['power']      = df['power'].ffill(limit=1)

# What ffill does:
# Before: [108, NaN, NaN, 110]
# After:  [108, 108, 108, 110]  (limit=3, all three gaps filled)
#
# With limit=1:
# Before: [108, NaN, NaN, 110]
# After:  [108, 108, NaN, 110]  (only the first gap filled)

## 7. Finding timestamp gaps (pause detection)

During auto-pauses the Garmin stops sending `record` messages, creating gaps in the timestamp sequence.

**Important:** always use `.total_seconds()` not `.seconds`. The `.seconds` attribute only returns the seconds component of the timedelta and resets at 1 hour — it silently gives wrong results on rides longer than 60 minutes.

In [ ]:
import datetime

# The .seconds vs .total_seconds() trap
delta = datetime.timedelta(hours=1, seconds=5)
print(delta.seconds)         # 5       <- WRONG for gaps, only the seconds component
print(delta.total_seconds()) # 3605.0  <- CORRECT, full duration in seconds

def find_gaps(timestamps):
    time_gap = []
    for count, timestamp in enumerate(timestamps):
        if count > 0:
            diff = timestamp - timestamps[count - 1]
            if diff.total_seconds() > 1:
                time_gap.append({
                    'start_time': timestamps[count - 1],
                    'end_time':   timestamp,
                    'gap':        diff.total_seconds()
                })
    return time_gap

## 8. Adding columns to a DataFrame

Assigning to a new column name creates it. Always work on a `.copy()` inside functions — modifying a DataFrame passed in as an argument can cause unexpected side effects on the caller's data.

In [ ]:
df = df.copy()  # defensive copy

# Add a boolean column
df['is_pause'] = False
df.loc[some_timestamp, 'is_pause'] = True

# Add a calculated column
df['hr_delta'] = df['heart_rate'] - 135.9

# f-strings make dynamic column names readable
signal = 'heart_rate'
df[f'delta_{signal}'] = df[signal] - baselines[signal]

## 9. Timedelta arithmetic — computing baseline windows

`pd.Timedelta` adds or subtracts durations from timestamps. This computes the start and end of the baseline window without needing exact matching timestamps to exist in the data.

Always guard against bad inputs — pandas silently clamps out-of-range timestamps rather than raising an error.

In [ ]:
def get_baseline_timestamps(timestamps, minute_start, minute_end):
    # Guard: negative values or end <= start produce nonsensical windows silently
    if minute_start < 0 or minute_end <= minute_start:
        raise ValueError('Invalid baseline window')
    ride_start = timestamps[0]
    return [
        ride_start + pd.Timedelta(minutes=minute_start),
        ride_start + pd.Timedelta(minutes=minute_end)
    ]

start, end = get_baseline_timestamps(df.index, 5, 15)
window = df.loc[start:end]  # safe even if exact timestamps don't exist in the index

## 10. Baselines as a dictionary — not a list

Storing baselines keyed by signal name makes access explicit and order-independent.

**Why not a list?** `baseline[0]` assumes heart_rate is always first. If someone calls `get_baselines(['power', 'heart_rate'], ...)` the values are in the wrong positions — silent wrong results.

In [ ]:
def get_baselines(parameters, df, minute_start, minute_end):
    ride_start   = df.index[0]
    window_start = ride_start + pd.Timedelta(minutes=minute_start)
    window_end   = ride_start + pd.Timedelta(minutes=minute_end)
    baselines = {}
    for p in parameters:
        baselines[p] = df[p][window_start:window_end].mean()
    return baselines

baselines = get_baselines(['heart_rate', 'power'], df, 5, 15)
# {'heart_rate': 135.9, 'power': 143.9}

print(baselines['heart_rate'])  # 135.9 — always correct regardless of input order
print(baselines['power'])       # 143.9

## 11. Iterating over a dictionary

When you loop over a dict you get the keys. Use the key to access the value.

In [ ]:
baselines = {'heart_rate': 135.9, 'power': 143.9}

for key in baselines:        # key = 'heart_rate', then 'power'
    value = baselines[key]   # value = 135.9, then 143.9
    print(key, value)

# Applied in add_delta_features:
def add_delta_features(df, baselines):
    df = df.copy()
    for b in baselines:
        df[f'delta_{b}'] = df[b] - baselines[b]
    return df

## 12. Rolling windows — mean and slope

A rolling window slides through the DataFrame one row at a time, looking back N rows and applying a function to that chunk.

**Why the first N rows are NaN:** the window is not yet full. At row 0 there are no previous rows to include. This is correct — do not fill these NaNs.

In [ ]:
# Rolling mean — built in
df['hr_rolling_mean'] = df['heart_rate'].rolling(300).mean()
# rows 0-298: NaN (window not yet full)
# row 299 onward: mean of the 300 rows ending at that point

# Rolling slope — needs a custom function via apply()
import numpy as np

def get_slope(window):
    # pandas passes each 300-row chunk as a numpy array automatically
    # np.arange(300) = [0, 1, 2, ..., 299]  <- relative position (x axis)
    # window = HR values                     <- the signal (y axis)
    # 1 = fit a straight line (y = slope*x + intercept)
    # [0] = take only the slope, not the intercept
    return np.polyfit(np.arange(len(window)), window, 1)[0]

df['hr_rolling_slope'] = df['heart_rate'].rolling(300).apply(get_slope)
# Positive = HR rising over that 5-minute window
# Negative = HR falling

## 13. How rolling().apply() works

`apply()` calls your function once per window position, passing the current chunk as a numpy array. You define what to do with each chunk — pandas handles the sliding and the calling.

```
rows 0-299  -> pandas extracts as numpy array -> calls get_slope(array) -> stores result at row 299
rows 1-300  -> pandas extracts as numpy array -> calls get_slope(array) -> stores result at row 300
rows 2-301  -> ...and so on
```

You never call `get_slope()` yourself. `apply()` on its own passes the entire column to your function. Combined with `rolling()`, it passes one chunk at a time.

## 14. np.polyfit — fitting a line to data

`np.polyfit(x, y, degree)` finds the best-fit polynomial through a set of points. Degree=1 fits a straight line and returns `[slope, intercept]`.

The slope is the gradient — how many units y changes per unit of x. In our case: bpm per second.

In [ ]:
import numpy as np

hr_values = [120, 122, 123, 125, 128]
x = np.arange(len(hr_values))  # [0, 1, 2, 3, 4]

result = np.polyfit(x, hr_values, 1)
print(result[0])  # ~1.9 — HR rising ~1.9 bpm per second
print(result[1])  # intercept — not used for drift detection

# Why np.arange and not real timestamps?
# polyfit needs plain numbers. Timestamps are datetime objects — polyfit
# cannot add a float to a datetime. It only cares about relative
# position (0, 1, 2...) not the actual clock time.

## 15. Handling infinity — zero power division

Dividing by zero produces `inf` or `-inf`. These corrupt rolling calculations. Replace with `NaN` to exclude those rows.

Both signs must be handled because the sign of the result depends on the sign of the numerator.

In [ ]:
ratio = df['heart_rate'] / df['power']  # zero power rows -> inf

ratio = ratio.replace([np.inf, -np.inf], np.nan)
# NaN is correct here — zero power means coasting, not pedaling.
# These rows carry no signal about cardiac drift and should be excluded.

## 16. Avoid inplace=True

`inplace=True` modifies the original object and returns `None` — but its behaviour is inconsistent across pandas versions. Always prefer explicit reassignment.

In [ ]:
# Avoid this — inplace=True is unpredictable across versions:
# df['col'].replace([np.inf, -np.inf], np.nan, inplace=True)

# Always prefer this — explicit and version-safe:
df['col'] = df['col'].replace([np.inf, -np.inf], np.nan)

## 17. Finding a row by timestamp and printing a window around it

`get_loc()` converts a timestamp to its integer position in the index, so you can slice a window of rows around it by position.

In [ ]:
# nlargest(n) returns the n largest values with their index labels
# Equivalent to: df['col'].sort_values(ascending=False).head(n)
peak_ts = df['heart_rate_slope'].nlargest(1).index[0]

# Convert timestamp to integer position
pos = df.index.get_loc(peak_ts)

# Print the 300-row window ending at that position
print(df[['heart_rate', 'heart_rate_slope']].loc[df.index[pos-300]:df.index[pos]])
# Verification: HR at the start of the window should be lower than at the end

## 18. The complete Week 3 pipeline

These functions must be called in this specific order. Calling them out of order produces wrong features with no error message. A future improvement is to wrap them in a single pipeline function to enforce the order.

In [ ]:
from fitparse import FitFile

activity = FitFile('your_activity.fit')

# 1. Parse and clean raw records
records = get_cleaned_records(activity)

# 2. Compute per-ride baselines — must come before deltas
baselines = get_baselines(['heart_rate', 'power'], records, 5, 15)

# 3. Add delta features (deviation from baseline)
records = add_delta_features(records, baselines)

# 4. Add rolling slope and mean (5-minute window = 300 rows)
records = add_slope_and_mean(records, ['heart_rate', 'power'], 300)

# 5. Add HR/power ratio delta
records = add_hr_power_delta_ratio(records, baselines)

# Future: wrap all steps in a pipeline function
# def build_feature_dataframe(fit_file, baseline_start=5, baseline_end=15, rolling_length=300):
#     records = get_cleaned_records(fit_file)
#     baselines = get_baselines(['heart_rate', 'power'], records, baseline_start, baseline_end)
#     records = add_delta_features(records, baselines)
#     records = add_slope_and_mean(records, ['heart_rate', 'power'], rolling_length)
#     records = add_hr_power_delta_ratio(records, baselines)
#     return records, baselines